# His-MMDM training data vs our cohort: patient overlap

We are considering His-MMDM (Li et al., *Advanced Science* 2026) as the diffusion model for the synthetic
condition. Before using it, this notebook checks whether any patient in our cohort (TCGA-GBM training/CV,
CPTAC-GBM external evaluation) also appears in the data His-MMDM was trained on. A shared patient would leak
information into the synthetic bags or the external evaluation.

It builds one patient list for each side, takes their union (outer join), and reports the overlap
(patients present on both sides). It downloads files and reads them; it does not change any data.

### What each His-MMDM source can show

| Source | IDs it carries | Covers |
|---|---|---|
| Zenodo record 12636449, `HMU_C_slides_metadata.csv` | Harbin hospital patient numbers | HMU-C lung primaries and metastases (released slides) |
| Zenodo record 12636449, `HMU_1st_slides_metadata.csv` | Harbin slide IDs (patient number derived) | HMU-1st glioma/meningioma H&E + IHC (released slides) |
| `demo_data.tar.gz` from the authors' S3 bucket | TCGA sample barcodes | the genomics (522 genes x 6,695 samples) and transcriptomics (4,461 x 7,941) tables the TCGA omics checkpoint looks up |
| `train_demo_TCGA.tar.gz` (optional) | TCGA slide IDs in patch file names | the demo training set |
| `scripts/dataset_specific_configurations/TCGA.sh` on GitHub | TCGA project names | the 19 class labels of the released TCGA checkpoints |

The authors have not published the full list of the 22,596 TCGA slides they trained on. The omics tables are the
largest released list of TCGA sample IDs, and the class list says which TCGA projects the training patches came
from. A TCGA patient belongs to exactly one project, so if TCGA-GBM is not one of the 19 classes, no TCGA-GBM
patient was a training image. The HMU IDs are hospital numbers in a different format, so an overlap with them
is not possible by construction; they are included so the union is complete.

**How to use:** run the cells top to bottom in the `gbm-mgmt` conda environment (or Colab). The first run
downloads the files into `ROOT/external/hismmdm/` (outside the repo) and checks their MD5s against the values
the authors published; later runs reuse them.

## Setup

Imports, and `load_manifest` from the repo's `data_prep/download.py`, which checks the frozen manifest's pinned
SHA-256 before anything reads it.

In [1]:
import hashlib
import io
import re
import sys
import tarfile
from pathlib import Path

import pandas as pd
import requests

REPO = Path("~/Bioinformatics/WSI/gbm-mgmt-wsi").expanduser()
ROOT = Path("~/Bioinformatics/WSI/data").expanduser()
CACHE = ROOT / "external" / "hismmdm"      # downloads and the output table land here, outside the repo
sys.path.insert(0, str(REPO / "data_prep"))
from download import DEFAULT_MANIFEST, load_manifest  # noqa: E402

### Settings

- `INCLUDE_TRAIN_DEMO`: also download and scan `train_demo_TCGA.tar.gz` (patch images; its size is printed first)
- `MAX_DOWNLOAD_GB`: refuse any single download larger than this
- `GDC_LOOKUP`: look up the GDC project of every TCGA patient found in His-MMDM's files

In [2]:
INCLUDE_TRAIN_DEMO = False
MAX_DOWNLOAD_GB = 5
GDC_LOOKUP = True

## Download the His-MMDM files

Each file is skipped when it is already in `CACHE` with the right MD5. The Zenodo MD5s are the ones shown on
the record page; the S3 MD5s are the ones pinned in the authors' `scripts/download.py`.

In [3]:
# Download the His-MMDM files that carry patient or sample IDs, verifying each MD5.
ZENODO = "https://zenodo.org/records/12636449/files"
S3 = "https://hismmdm.s3.amazonaws.com"
SOURCES = {  # local name: (url, md5 published by the authors)
    "HMU_C_slides_metadata.csv": (f"{ZENODO}/HMU_C_slides_metadata.csv?download=1",
                                  "8436297fe471e28bce4c0a9b562af370"),
    "HMU_1st_slides_metadata.csv": (f"{ZENODO}/HMU_1st_slides_metadata.csv?download=1",
                                    "d987b0335de008e18e522644c1ba8348"),
    "demo_data.tar.gz": (f"{S3}/inference/demo_data.tar.gz", "d3f116d9e4cec7ef88c360c94f782de7"),
}
if INCLUDE_TRAIN_DEMO:
    SOURCES["train_demo_TCGA.tar.gz"] = (f"{S3}/train/train_demo_TCGA.tar.gz",
                                         "21c554f3d68c188a667d0023bb133873")


def md5_of(path):
    digest = hashlib.md5()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def fetch(name, url, md5):
    path = CACHE / name
    if path.exists() and md5_of(path) == md5:
        return path
    size = int(requests.head(url, allow_redirects=True, timeout=60).headers.get("Content-Length", 0))
    if size > MAX_DOWNLOAD_GB * 1e9:
        raise RuntimeError(f"{name}: {size / 1e9:.1f} GB > MAX_DOWNLOAD_GB")
    print(f"downloading {name} ({size / 1e6:,.1f} MB)" if size else f"downloading {name} (size unknown)")
    part = path.with_name(name + ".part")
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with open(part, "wb") as handle:
            for block in response.iter_content(1 << 20):
                handle.write(block)
    found = md5_of(part)
    if found != md5:
        raise RuntimeError(f"{name}: md5 {found} != {md5}")
    part.rename(path)
    return path


CACHE.mkdir(parents=True, exist_ok=True)
FILES = {name: fetch(name, url, md5) for name, (url, md5) in SOURCES.items()}
for name, path in FILES.items():
    print(f"{name:30s} {path.stat().st_size / 1e6:10,.1f} MB   md5 ok")

downloading HMU_C_slides_metadata.csv (size unknown)
downloading HMU_1st_slides_metadata.csv (size unknown)
downloading demo_data.tar.gz (142.0 MB)
HMU_C_slides_metadata.csv             0.0 MB   md5 ok
HMU_1st_slides_metadata.csv           0.0 MB   md5 ok
demo_data.tar.gz                    142.0 MB   md5 ok


## Our cohort

One row per patient from the frozen manifest: 151 TCGA-GBM and 83 CPTAC-GBM patients.

In [4]:
# Our cohort: one row per patient from the frozen manifest (load_manifest checks the pinned SHA-256).
_, manifest = load_manifest(DEFAULT_MANIFEST)
ours = (pd.DataFrame(manifest["files"])
        .groupby(["source", "patient_id"]).size().rename("slides").reset_index()
        .rename(columns={"source": "id_type"})
        .assign(our_cohort=lambda df: df.id_type.map({"TCGA": "TCGA-GBM", "CPTAC": "CPTAC-GBM"})))
ours.groupby("our_cohort").agg(patients=("patient_id", "size"), slides=("slides", "sum"))

,patients,slides
our_cohort,,
CPTAC-GBM,83,204
TCGA-GBM,151,320


## Matching IDs

A TCGA patient is the first 12 characters of a barcode (`TCGA-TSS-PART`, e.g. `TCGA-02-0003` from
`TCGA-02-0003-01`); CPTAC-GBM patients are `C3L-` or `C3N-` plus five digits. Some tools rewrite `-` as `.`
or `_`, so all three separators are accepted and normalised to `-`.

The positive control runs the same matcher over the raw text of our manifest and must recover every one of our
234 patients. That way an empty overlap means the two lists share no IDs, not that the matcher missed them.

In [5]:
# ID matcher, plus a positive control on our own manifest.
TCGA_ID = re.compile(r"TCGA[-._]([A-Z0-9]{2})[-._]([A-Z0-9]{4})", re.I)
CPTAC_ID = re.compile(r"(C3[LN])[-._](\d{5})", re.I)


def find_ids(text):
    # {(id_type, patient_id)} for every TCGA or CPTAC patient ID in text
    tcga = {("TCGA", f"TCGA-{tss}-{part}".upper()) for tss, part in TCGA_ID.findall(text)}
    cptac = {("CPTAC", f"{prefix}-{number}".upper()) for prefix, number in CPTAC_ID.findall(text)}
    return tcga | cptac


expected = set(zip(ours.id_type, ours.patient_id))
recovered = find_ids(DEFAULT_MANIFEST.read_text())
missing = sorted(expected - recovered)
print(f"positive control: recovered {len(expected & recovered)} / {len(expected)} of our patients")
assert not missing, missing[:5]

positive control: recovered 234 / 234 of our patients


## His-MMDM: in-house Harbin cohorts (Zenodo)

`HMU_C` has a `patient_id` column. `HMU_1st` has no patient column, so the patient number is taken as the first
field of the slide ID. The cell checks that this rule holds in `HMU_C`, where both columns exist. Both files
are also scanned for anything shaped like a TCGA or CPTAC ID.

In [6]:
# His-MMDM in-house cohorts: Harbin hospital patient numbers, namespaced so they cannot collide with TCGA/CPTAC IDs.
hmu_c = pd.read_csv(FILES["HMU_C_slides_metadata.csv"], dtype=str)
hmu_1st = (pd.read_csv(FILES["HMU_1st_slides_metadata.csv"], dtype=str)
           .rename(columns={"Unnamed: 0": "slide_id"}))
first_field = hmu_c.slide_id.str.split("_").str[0]
print(f"HMU-C slides whose first slide-ID field is the patient_id: {(first_field == hmu_c.patient_id).mean():.1%}")
hmu_1st["patient_id"] = hmu_1st.slide_id.str.split("_").str[0]

hmu = pd.concat([
    pd.DataFrame({"id_type": "HMU", "patient_id": "HMU-C:" + hmu_c.patient_id, "hismmdm_source": "HMU-C (Zenodo)"}),
    pd.DataFrame({"id_type": "HMU", "patient_id": "HMU-1st:" + hmu_1st.patient_id,
                  "hismmdm_source": "HMU-1st (Zenodo)"}),
]).drop_duplicates()
print(f"HMU-C: {hmu_c.patient_id.nunique()} patients, {len(hmu_c)} slides, sites {sorted(hmu_c.tissue_origin.unique())}")
print(f"HMU-1st: {hmu_1st.patient_id.nunique()} patients, {len(hmu_1st)} slides, "
      f"types {sorted(hmu_1st.slide_type.unique())}")
zenodo_text = "\n".join(FILES[name].read_text() for name in ("HMU_C_slides_metadata.csv", "HMU_1st_slides_metadata.csv"))
print("TCGA/CPTAC-shaped IDs in the Zenodo files:", sorted(find_ids(zenodo_text)) or "none")

HMU-C slides whose first slide-ID field is the patient_id: 100.0%
HMU-C: 331 patients, 478 slides, sites ['brain', 'lung', 'lymph_node']
HMU-1st: 206 patients, 210 slides, types ['HE', 'IHC']
TCGA/CPTAC-shaped IDs in the Zenodo files: none


## His-MMDM: TCGA (released archives)

Every member of each archive is read in memory; nothing is extracted to disk.

- **CSV/TSV:** the header row and first column only. The omics tables are genes x samples, so the sample barcodes
  are the header (`pd.read_csv(..., index_col=0)` in the authors' code).
- **Other small text files:** the whole text.
- **Everything else** (images, `.pkl`, checkpoints): the file name only. Pickles are never loaded, since loading one
  runs code from the file.

In [7]:
# Scan every member of the released archives for TCGA/CPTAC IDs, without extracting anything.
TEXT_SUFFIXES = {".txt", ".json", ".yaml", ".yml", ".sh"}


def member_text(archive, member):
    # the parts of one archive member that are searched for IDs
    texts = [member.name]
    suffix = Path(member.name).suffix.lower()
    if suffix in {".csv", ".tsv"}:
        separator = "," if suffix == ".csv" else "\t"
        with archive.extractfile(member) as handle:
            lines = io.TextIOWrapper(handle, encoding="utf-8", errors="replace")
            texts.append(next(lines, ""))
            texts.extend(line.split(separator, 1)[0] for line in lines)
    elif suffix in TEXT_SUFFIXES and member.size < 50e6:
        with archive.extractfile(member) as handle:
            texts.append(handle.read().decode("utf-8", "replace"))
    return "\n".join(texts)


hits = []
for name, path in FILES.items():
    if not name.endswith(".tar.gz"):
        continue
    with tarfile.open(path, "r:gz") as archive:
        for member in archive:
            if member.isfile():
                hits += [{"id_type": kind, "patient_id": pid, "archive": name, "member": member.name}
                         for kind, pid in find_ids(member_text(archive, member))]
hits = pd.DataFrame(hits, columns=["id_type", "patient_id", "archive", "member"])

print(hits.groupby("id_type").patient_id.nunique().rename("unique patients").to_string())
print("\nMembers with the most patient IDs:")
hits.groupby(["archive", "member"]).patient_id.nunique().sort_values(ascending=False).head(10)

id_type
TCGA    7656

Members with the most patient IDs:


archive           member                                                                                                                    
demo_data.tar.gz  demo_data/TCGA-general/transcriptomics_table.csv                                                                              7242
                  demo_data/TCGA-general/genomics_table.csv                                                                                     6695
                  demo_data/TCGA-genomics/TCGA-COAD__TCGA-A6-2671-11A-01-TS1.f8122238-e900-407d-b9f8-a1d7ba677bb0.177_18384_11169.png              1
                  demo_data/TCGA-genomics/TCGA-COAD__TCGA-A6-2672-01A-01-BS1.aab0019c-99f1-4580-9b12-b5b81e4e8346.3595_17176_8240.png              1
                  demo_data/TCGA-primary_sites/TCGA-COAD__TCGA-A6-4105-01A-02-BS2.e609908d-b032-48ff-a301-e8810dfb7552.6362_27376_11296.png        1
                  demo_data/TCGA-primary_sites/TCGA-KICH__TCGA-KL-8323-11A-01-TS1.03244a58-76a3-4550-8065-2a7e765f

## His-MMDM: TCGA classes of the released checkpoints

The class labels come from the authors' config for the TCGA checkpoints. If TCGA-GBM is not among them, no TCGA-GBM
slide was a training image for those checkpoints.

In [8]:
# The TCGA projects the released TCGA checkpoints were trained on, read from the authors' config on GitHub.
CONFIG_URL = ("https://raw.githubusercontent.com/lzx325/His-MMDM/main/"
              "scripts/dataset_specific_configurations/TCGA.sh")
config = requests.get(CONFIG_URL, timeout=60)
config.raise_for_status()
class_mapping = re.search(r'class_mapping="([^"]+)"', config.text).group(1)
classes = [item.split(":")[0] for item in class_mapping.split(",")]
print(f"{len(classes)} classes: {', '.join(classes)}")
print(f"TCGA-GBM is a class: {'TCGA-GBM' in classes}   TCGA-LGG is a class: {'TCGA-LGG' in classes}")

19 classes: TCGA-BLCA, TCGA-BRCA, TCGA-COAD, TCGA-ESCA, TCGA-HNSC, TCGA-KICH, TCGA-KIRC, TCGA-KIRP, TCGA-LIHC, TCGA-LUAD, TCGA-LUSC, TCGA-OV, TCGA-PAAD, TCGA-PRAD, TCGA-READ, TCGA-SARC, TCGA-STAD, TCGA-THCA, TCGA-UCEC
TCGA-GBM is a class: False   TCGA-LGG is a class: False


## Union and overlap

Outer join of the two patient lists on (ID type, patient ID). `present_in` says which side each patient is on;
rows with `both` are the overlap. The full table is saved to `CACHE/patient_union.csv`.

In [9]:
# Union of our patients and His-MMDM's (outer join); "both" rows are the overlap.
tcga_side = (hits.assign(hismmdm_source=lambda df: df.archive + ": " + df.member.str.split("/").str[-1])
             .groupby(["id_type", "patient_id"]).hismmdm_source
             .agg(lambda sources: "; ".join(sorted(set(sources)))).reset_index())
theirs = pd.concat([tcga_side, hmu], ignore_index=True)

union = ours.merge(theirs, on=["id_type", "patient_id"], how="outer", indicator="present_in")
union["present_in"] = union.present_in.map({"left_only": "ours only", "right_only": "His-MMDM only", "both": "both"})
union.to_csv(CACHE / "patient_union.csv", index=False)

overlap = union[union.present_in == "both"]
print(f"union: {len(union):,} patients   overlap: {len(overlap)}")
for cohort in ("TCGA-GBM", "CPTAC-GBM"):
    print(f"  {cohort}: {(overlap.our_cohort == cohort).sum()} of {(ours.our_cohort == cohort).sum()} patients "
          f"also in His-MMDM's files")
display(union.groupby(["id_type", "present_in"]).size().unstack(fill_value=0))
overlap

union: 8,427 patients   overlap: 0
  TCGA-GBM: 0 of 151 patients also in His-MMDM's files
  CPTAC-GBM: 0 of 83 patients also in His-MMDM's files


present_in,ours only,His-MMDM only
id_type,,
CPTAC,83,0
HMU,0,537
TCGA,151,7656


,id_type,patient_id,slides,our_cohort,hismmdm_source,present_in


## GDC project of each TCGA patient

Looks up every TCGA patient on either side by case `submitter_id` (batches of 500), using the
[GDC API](https://docs.gdc.cancer.gov/API/Users_Guide/Search_and_Retrieval/). Our TCGA patients are the control:
all 151 should come back as TCGA-GBM. If any His-MMDM patient comes back as TCGA-GBM or TCGA-LGG, look at it in
the overlap table above.

In [10]:
# GDC project of every TCGA patient on either side; ours should all be TCGA-GBM.
GDC_CASES = "https://api.gdc.cancer.gov/cases"


def gdc_projects(patient_ids, batch=500):
    rows = []
    for start in range(0, len(patient_ids), batch):
        chunk = patient_ids[start:start + batch]
        payload = {"filters": {"op": "in", "content": {"field": "submitter_id", "value": chunk}},
                   "fields": "submitter_id,project.project_id", "format": "json", "size": str(len(chunk))}
        response = requests.post(GDC_CASES, json=payload, timeout=120)
        response.raise_for_status()
        rows += [{"patient_id": hit["submitter_id"], "project": hit.get("project", {}).get("project_id")}
                 for hit in response.json()["data"]["hits"]]
    return pd.DataFrame(rows, columns=["patient_id", "project"])


if GDC_LOOKUP:
    tcga = union[union.id_type == "TCGA"]
    projects = gdc_projects(sorted(tcga.patient_id))
    tcga = tcga.merge(projects, on="patient_id", how="left").fillna({"project": "not found"})
    table = pd.crosstab(tcga.project, tcga.present_in, margins=True)
    flagged = tcga[(tcga.present_in != "ours only") & tcga.project.isin(["TCGA-GBM", "TCGA-LGG"])]
    print(f"His-MMDM TCGA patients in TCGA-GBM or TCGA-LGG: {len(flagged)}")
    display(table)

His-MMDM TCGA patients in TCGA-GBM or TCGA-LGG: 0


present_in,ours only,His-MMDM only,All
project,,,
TCGA-BLCA,0,412,412
TCGA-BRCA,0,1098,1098
TCGA-COAD,0,322,322
TCGA-ESCA,0,185,185
TCGA-GBM,151,0,151
TCGA-HNSC,0,528,528
TCGA-KICH,0,66,66
TCGA-KIRC,0,535,535
TCGA-KIRP,0,291,291


## Result

*Fill in after running:* overlap counts for TCGA-GBM and CPTAC-GBM, the positive-control line, whether TCGA-GBM/LGG
are among the 19 classes, and the GDC project breakdown.

**Reading the result**

- **Overlap 0, GBM/LGG not among the classes:** no patient in our cohort is in anything His-MMDM released, and no TCGA
  glioma project was a training class. The released TCGA checkpoints are safe on leakage. The HMU-1st glioma
  checkpoint is a separate question: its patients are Harbin hospital patients, so it cannot share patients with
  TCGA or CPTAC.
- **Any overlap:** check where each shared patient was found (`hismmdm_source`) before acting on it. The paper
  ran TCGA GBM/LGG *frozen sections* through the already-trained cryo-to-FFPE checkpoint for a CHIEF IDH1 test.
  That was inference, so those patients can appear in released files without having trained the model. A shared
  patient in training data (a training archive, or a TCGA-GBM class) is real leakage: list them in the proposal
  and either drop them from the cohort or rule out that checkpoint.
- **Limit:** this checks the IDs the authors released. The exact list of 22,596 training slides is not public; the
  class list is what rules out TCGA-GBM as a training project.